# 🧬 Connect AI — 장기 기억 학습 (Unsloth)
내 1인 기업 지식을 모델 **가중치에 체득**시킵니다. 위 메뉴 **런타임 → 모두 실행**만 누르면 됩니다 (무료 T4 GPU).
- 데이터셋: `WonseokJayJung/aimentorjay_dataset` (단기 지식 → conversations Q&A)
- 베이스 모델: `unsloth/gemma-4-E2B-it`  ← *내가 쓰는 모델로 바꿔도 됩니다 (누적 학습)*
- 결과 모델: `WonseokJayJung/테스트용` (GGUF — LM Studio/Ollama에 바로 로드)
- 설정: rank 16/alpha 32 · dropout 0 · lr 0.0003 · steps 142 · seq 1024 · linear · 양자화 q4_k_m (데이터 71개)


In [ ]:
%%capture
import os, re
!pip install unsloth
!pip install --no-deps "xformers<0.0.30" trl peft accelerate bitsandbytes datasets


In [ ]:
from unsloth import FastModel
import torch
model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E2B-it",
    dtype = None, max_seq_length = 1024,
    load_in_4bit = True, full_finetuning = False,
)
print("✅ 베이스 모델 로딩 완료")


In [ ]:
# LoRA — 전체의 1% 미만만 학습(메모리↓, 페르소나·핵심지식엔 충분)
model = FastModel.get_peft_model(
    model, finetune_language_layers=True, finetune_attention_modules=True,
    finetune_mlp_modules=True, finetune_vision_layers=False,
    r = 16, lora_alpha = 32, lora_dropout = 0, bias = "none", random_state = 3407,
)


## 📦 단기 지식 데이터셋 불러오기 (conversations Q&A)
Connect AI 앱이 업로드한 데이터셋. 각 행 = `{conversations:[{user},{assistant}]}`


In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, standardize_data_formats
ds = load_dataset("WonseokJayJung/aimentorjay_dataset", data_files="connect-ai-brain.jsonl", split="train")
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
def fmt(ex):
    texts = [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False).removeprefix("<bos>") for c in ex["conversations"]]
    return {"text": texts}
ds = ds.map(fmt, batched=True)
print("데이터 개수:", len(ds)); print(ds[0]["text"][:400])


In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model, tokenizer = tokenizer, train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 1, gradient_accumulation_steps = 4,
        warmup_steps = 5, max_steps = 142, learning_rate = 0.0003,
        logging_steps = 1, optim = "adamw_8bit", weight_decay = 0.001,
        lr_scheduler_type = "linear", seed = 3407, report_to = "none",
    ),
)


In [ ]:
# 🎭 응답(assistant)만 학습 — 질문 패턴은 마스킹(효율↑·품질↑)
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer, instruction_part="<start_of_turn>user\n", response_part="<start_of_turn>model\n")
print("✅ 학습 준비 완료 — 다음 셀에서 학습 시작")


In [ ]:
trainer_stats = trainer.train()
print("🎉 학습 완료! 최종 loss:", round(trainer_stats.training_loss, 4))
print("💡 loss 0.2~0.4면 sweet spot. 너무 낮으면(<0.1) 과적합 — max_steps 줄이세요.")


## 💾 GGUF로 저장 (LM Studio/Ollama용)
아래 셀 실행 후 토큰 칸에 HuggingFace **write 토큰**을 붙여넣으세요.


In [ ]:
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
# 내 모델 = 장기 기억. q4_k_m GGUF 로 저장 + HF 업로드
model.push_to_hub_gguf("WonseokJayJung/테스트용", tokenizer, quantization_method="q4_k_m", token=True)
print("✅ 완료! huggingface.co/WonseokJayJung/테스트용 에서 .gguf 다운로드 → LM Studio/Ollama 로드 → ⚙️설정에서 선택")
